# Project 03: Gibbs Sampling for Motif Finding

## Purpose
Implement Gibbs sampling MCMC to discover enriched motifs from a set of DNA sequences (NRF1 ChIP-seq promoter regions). Each sequence contains one instance of the motif of length k=10.


## Imports
We need numpy for core logic, bamnostic for BAM parsing (optional fallback to synthetic data), and seqlogo for visualization.

In [ ]:
import numpy as np
import random
import sys
sys.path.append('.')
try:
    import bamnostic as bs
    print("bamnostic available")
except ImportError:
    print("bamnostic not installed - using synthetic fallback")
    bs = None

try:
    import seqlogo
    print("seqlogo available")
except ImportError:
    print("seqlogo not available - will plot with matplotlib/logomaker")
    seqlogo = None

from motif_ops import build_pfm, build_pwm, score_kmer, score_sequence, pfm_ic, reverse_complement, pfm_to_pwm
from gibbs_sampler import GibbsMotifFinder


## Data ingestion
Original project uses BAM file of NRF1 binding. We attempt to read via bamnostic, otherwise generate synthetic promoter data with implanted motif.
NRF1 known motif is GC-rich: GCGCATGCGC is common, but we use k=10.

In [ ]:
def load_seqs_from_bam(bam_path):
    """Example using bamnostic - returns list of sequences"""
    if bs is None:
        return None
    # Placeholder: actual implementation would parse BAM and extract flanking sequences
    # For this notebook we skip because BAM not provided
    return None

def generate_nrf1_like_data(num_seqs=30, seq_len=100, k=10, seed=1):
    """Generate NRF1-like GC-rich motif data for testing"""
    np.random.seed(seed)
    # NRF1 consensus: GCGCATGCGC (or variations)
    true_motif = "GCGCATGCGC"[:k]  # trim to k
    seqs = []
    for _ in range(num_seqs):
        bg = ''.join(np.random.choice(list('ACGT'), size=seq_len))
        pos = np.random.randint(0, seq_len - k)
        # introduce 15% mutation
        mutated = ''.join([b if np.random.rand()>0.15 else np.random.choice(list('ACGT')) for b in true_motif])
        seq = bg[:pos] + mutated + bg[pos+k:]
        seqs.append(seq)
    return seqs, true_motif

seqs, true_motif = generate_nrf1_like_data(num_seqs=50, seq_len=100, k=10, seed=42)
print(f"Loaded {len(seqs)} seqs, example: {seqs[0][:60]}...")
print(f"True motif implanted: {true_motif}")


## Gibbs Sampler implementation
Pseudocode from instructions:
```
GibbsMotifFinder(DNA, k):
    random pick of k-length sequences from each line of DNA as Motifs
    for j <- 1 to 10000 or Motifs stops changing
        i <- Random(N)
        PWM <- PWM constructed from all Motifs except Motifi
        Motifi <- select position m from PWM-scored k-mers in DNAi in probabilistic fashion
    return PFM
```
Key considerations implemented:
- Both strands scored via reverse complement
- PWM built with pseudocount=1 to avoid zero probabilities
- Sampling via numpy.random.choice with softmax over exp(scores)
- Convergence: no change for 10 full sweeps OR 10000 iterations
- Information content tracked via pfm_ic()


In [ ]:
# Run the gibbs sampler - driver program from assignment (must work without alteration)
promoter_pfm = GibbsMotifFinder(seqs, 10)
print("Final PFM shape:", promoter_pfm.shape)
print(promoter_pfm)
print("IC:", pfm_ic(promoter_pfm))

# Consensus
BASES = ['A','C','G','T']
consensus = ''.join([BASES[np.argmax(promoter_pfm[:,j])] for j in range(promoter_pfm.shape[1])])
print("Recovered consensus:", consensus)


## Plot sequence logo
Uses seqlogo.CompletePm if available, otherwise matplotlib.

In [ ]:
import matplotlib.pyplot as plt

if seqlogo is not None:
    try:
        # seqlogo expects pfm transposed: CompletePm(pfm = promoter_pfm.T)
        logo = seqlogo.seqlogo(seqlogo.CompletePm(pfm = promoter_pfm.T))
        display(logo)
    except Exception as e:
        print("seqlogo failed:", e)
        # fallback
        seqlogo = None

if seqlogo is None:
    # fallback using logomaker if available, else bar plot
    try:
        import logomaker
        import pandas as pd
        pfm = promoter_pfm
        # convert to freq df
        col_sums = pfm.sum(axis=0)
        freq = pfm / col_sums
        df = pd.DataFrame(freq.T, columns=['A','C','G','T'])
        logo = logomaker.Logo(df)
        plt.show()
    except ImportError:
        # simple bar plot per position
        fig, axes = plt.subplots(2,5, figsize=(12,4), sharey=True)
        axes = axes.flatten()
        for j in range(10):
            axes[j].bar(['A','C','G','T'], promoter_pfm[:,j])
            axes[j].set_title(f"pos {j}")
        plt.tight_layout()
        plt.show()


## Information Content progression
IC = pfm_ic(pfm) should slowly increase and plateau.

In [ ]:
# Re-run with tracking
import importlib
import motif_ops
importlib.reload(motif_ops)

# Modified version that returns history
def GibbsMotifFinder_with_history(seqs, k, seed=42, max_iter=10000):
    np.random.seed(seed)
    seqs = [s.upper().replace('U','T') for s in seqs]
    n = len(seqs)
    motifs = []
    positions = []
    for seq in seqs:
        start = np.random.randint(0, len(seq)-k+1)
        positions.append(start)
        motifs.append(seq[start:start+k])
    ic_history = []
    last_motifs = motifs.copy()
    no_change = 0
    for iteration in range(max_iter):
        i = np.random.randint(0,n)
        excl = motifs[:i] + motifs[i+1:]
        pfm_excl = motif_ops.build_pfm(excl, pseudocount=0)
        pwm = motif_ops.build_pwm(pfm_excl, pseudocount=1.0)
        seq_i = seqs[i]
        L = len(seq_i)-k+1
        scores = []
        for j in range(L):
            kmer = seq_i[j:j+k]
            if 'N' in kmer:
                scores.append(-np.inf)
                continue
            rc = motif_ops.reverse_complement(kmer)
            scores.append(max(motif_ops.score_kmer(pwm,kmer), motif_ops.score_kmer(pwm,rc)))
        scores = np.array(scores)
        finite = np.isfinite(scores)
        max_s = np.max(scores[finite]) if np.any(finite) else 0
        exp_s = np.exp(scores-max_s)
        exp_s[~finite]=0
        probs = exp_s/exp_s.sum() if exp_s.sum()!=0 else np.ones(L)/L
        new_pos = np.random.choice(L,p=probs)
        kmer_new = seq_i[new_pos:new_pos+k]
        rc_new = motif_ops.reverse_complement(kmer_new)
        motifs[i] = rc_new if motif_ops.score_kmer(pwm,rc_new) > motif_ops.score_kmer(pwm,kmer_new) else kmer_new
        positions[i]=new_pos
        if iteration % n ==0:
            if motifs == last_motifs:
                no_change+=1
            else:
                no_change=0
                last_motifs = motifs.copy()
            ic = motif_ops.pfm_ic(motif_ops.build_pfm(motifs))
            ic_history.append(ic)
            if no_change>=10:
                break
    return motif_ops.build_pfm(motifs), ic_history

pfm_hist, ic_hist = GibbsMotifFinder_with_history(seqs,10)
plt.plot(ic_hist)
plt.xlabel("sweeps")
plt.ylabel("Information Content (bits)")
plt.title("Gibbs sampler convergence")
plt.show()
